In [1]:
import os
import sys
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

In [2]:
pd.set_option("display.expand_frame_repr", False)  # запрещаем перенос строк
import wquantiles  # библиотека для вычисления взвешенных статистик
from tqdm import tqdm

In [ ]:
adata = sc.read_h5ad("../adata_all_harm_ribo.h5ad")
# contains raw
obs = pd.read_csv(
    ".././0_adata obs - reduced and filt.csv",
    index_col=0,
)
adata = adata[obs.index].copy()
adata.obs = obs.loc[adata.obs_names].copy()
sc.pp.filter_genes(adata, min_cells=1)

In [4]:
obs.head()

,GSE_id,GSM_id,Age,Tissue_global,cell_type_final,Age_bin,cell_type,n_expr_genes
AAACCTGAGGCCGAAT-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,Fibroblast,21,Fibroblasts,950
AAACCTGAGTTAAGTG-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,CD8T,21,CD8 T cells,1247
AAACCTGGTCTTCTCG-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,VE,21,Vein endothelium,724
AAACCTGGTGCAACTT-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,DC,21,Dendritic cells,1888
AAACCTGTCAGCGACC-1-GSM3758115,GSE130973,GSM3758115,25.0,Skin,CD4T,21,CD4 T cells,721


In [5]:
adata

AnnData object with n_obs × n_vars = 1253024 × 43059
    obs: 'GSE_id', 'GSM_id', 'Age', 'Tissue_global', 'cell_type_final', 'Age_bin', 'cell_type', 'n_expr_genes'
    var: 'n_cells'
    uns: 'Tissue_global_colors'
    obsm: 'X_scANVI', 'X_scVI', 'X_umap', '_scvi_extra_categorical_covs'

### Downsampling

In [7]:
import scipy.sparse as sp
import warnings

In [8]:
sid_genes = {
    "SID1": [
        "CCND1",
        "CDKN1A",
        "PCNA",
        "CDK6",
        "ZMAT3",
        "EGR1",
        "BHLHE40",
        "MYLK",
        "EDN1",
        "F3",
        "PCK2",
        "BTG2",
        "TRIM22",
        "OSTM1",
        "DEPDC1",
        "NCEH1",
        "TP53INP1",
        "PRICKLE2",
        "PGM2L1",
    ],
    "SID2": [
        "CCND1",
        "HMGB2",
        "CKB",
        "CDKN2C",
        "EDN1",
        "FANCE",
        "PLAT",
        "ENC1",
        "SV2A",
        "PHGDH",
        "ANGPTL4",
        "MFSD1",
        "CDCA3",
    ],
    "SID3": [
        "CDK1",
        "LMNB1",
        "SOD2",
        "CCND1",
        "FGF2",
        "GDF15",
        "IGFBP3",
        "IGFBP5",
        "MKI67",
        "SERPINE1",
        "TOP2A",
        "CCNE1",
        "CBX4",
        "PRKCD",
        "CDKN2D",
        "CDK6",
        "SLC16A7",
        "RNASET2",
        "BUB1B",
        "TAGLN",
        "S100A6",
        "RXRB",
        "RRM2",
        "RING1",
        "CDKN2AIP",
        "PLAUR",
        "MYBL2",
        "MAP3K5",
        "SMAD1",
        "JAK2",
        "HNRNPA1",
        "HELLS",
        "NPTXR",
        "ETS2",
        "EGR1",
        "DUSP6",
        "DUSP4",
        "E2F7",
        "CKB",
        "UBE2C",
        "EHMT2",
        "NDRG1",
        "CD55",
        "ICAM3",
        "ITGA2",
        "BHLHE40",
        "HDAC4",
        "ING2",
        "MYLK",
        "PRKCH",
        "RUNX1",
        "SGK1",
        "TGFB1I1",
        "AMPD3",
        "SLC25A4",
        "AOX1",
        "BARD1",
        "CALD1",
        "CTSC",
        "CKS2",
        "COL6A3",
        "COL15A1",
        "SLC31A2",
        "DNASE1L1",
        "DPYSL3",
        "EDN1",
        "EMP2",
        "EPAS1",
        "F3",
        "FAT1",
        "FKBP5",
        "FLNC",
        "FN1",
        "STMN1",
        "LOX",
        "LOXL2",
        "LRP1",
        "MATN2",
        "MDK",
        "MEST",
        "MXI1",
        "PCOLCE",
        "SERPINF1",
        "PFKP",
        "PLAT",
        "PRPS1",
        "PTPRK",
        "PTX3",
        "SAT1",
        "SLC1A1",
        "SOX4",
        "SVIL",
        "SYT1",
        "TIMP3",
        "TMPO",
        "ZNF22",
        "BTG2",
        "CDC7",
        "STC2",
        "TNFRSF10D",
        "P4HA2",
        "SYNGR2",
        "SYNGR1",
        "DHRS3",
        "SLIT2",
        "ARHGAP29",
        "KIF23",
        "ATP6V1G1",
        "SH3PXD2A",
        "DLGAP5",
        "HEPH",
        "KIF14",
        "TROAP",
        "EDIL3",
        "KIF20A",
        "CITED2",
        "NDC80",
        "PGRMC2",
        "PLK4",
        "RAB40B",
        "CNTRL",
        "DNAJB4",
        "CIT",
        "DUSP10",
        "OIP5",
        "COBLL1",
        "NCAPH",
        "PLA2G15",
        "KIF4A",
        "LMOD1",
        "MPC2",
        "CYFIP2",
        "TSPAN13",
        "TNFRSF21",
        "EML4",
        "SLC43A3",
        "ARHGEF3",
        "ANGPTL4",
        "SCARA3",
        "DTL",
        "RAB6B",
        "CYB5R1",
        "GNG2",
        "ANLN",
        "MANSC1",
        "CEP55",
        "MCM10",
        "PLXNA3",
        "CDK5RAP2",
        "SULF2",
        "EMC7",
        "OLFML3",
        "PLSCR4",
        "KIAA1191",
        "NIPAL3",
        "SPC25",
        "RHOU",
        "FAM111A",
        "CLSPN",
        "PERP",
        "OGFRL1",
        "SCD5",
        "ARMC9",
        "FAM214B",
        "COLEC12",
        "BRIP1",
        "SGIP1",
        "MED30",
        "SLFN11",
        "ZNF300",
        "FANK1",
        "TIFA",
        "AHNAK2",
        "PIK3IP1",
        "FBXO32",
        "CCDC58",
        "FAM43A",
        "GPR155",
        "ZNF610",
        "ZNF367",
        "OTUD1",
        "ASPM",
        "GAS2L3",
        "NCKAP5",
        "FAM111B",
        "ARHGEF37",
    ],
    "SID4": [
        "CDKN2A",
        "CDKN2B",
        "GDF15",
        "SERPINE1",
        "CCNE1",
        "TBX2",
        "CEBPB",
        "ARHGAP18",
        "RING1",
        "MAPK10",
        "PLAUR",
        "IGF1R",
        "HSPB1",
        "CABIN1",
        "ETS2",
        "ETS1",
        "CLU",
        "BMP2",
        "GEM",
        "INHBA",
        "MMP14",
        "ARPC1B",
        "BAG3",
        "BHLHE40",
        "DHCR24",
        "SENP7",
        "SGK1",
        "ARF4",
        "STS",
        "CD81",
        "CTSC",
        "COL6A3",
        "COL8A1",
        "DPYSL3",
        "EMP2",
        "EPAS1",
        "F3",
        "FN1",
        "IFI6",
        "HAS2",
        "ITGAV",
        "ITGB5",
        "LAMC1",
        "MEST",
        "SERPINF1",
        "PPP1R3C",
        "SAT1",
        "TAP1",
        "TGM2",
        "SLC7A5",
        "ADAM19",
        "TNFRSF10D",
        "SPHK1",
        "SLC5A6",
        "SYNGR2",
        "DHRS3",
        "SNPH",
        "CPQ",
        "BLCAP",
        "PRSS23",
        "CIT",
        "IRAK3",
        "FILIP1L",
        "COBLL1",
        "TBC1D9",
        "OSTM1",
        "GNG2",
        "SAMD9",
        "ERCC6L",
        "CENPJ",
        "FIGNL1",
        "PERP",
        "TMBIM1",
        "PIP4K2C",
        "BORA",
        "PGBD1",
        "ARRDC4",
        "ZNF300",
        "HTRA3",
        "CYP2U1",
        "FBXO32",
        "FRMD6",
        "SLC43A2",
        "FAM43A",
        "LACC1",
        "CCDC138",
        "DCBLD1",
    ],
    "SID5": [
        "LMNB1",
        "CCL2",
        "GDF15",
        "IGFBP3",
        "MKI67",
        "SERPINE1",
        "TWIST1",
        "CCNE1",
        "PLAUR",
        "PIM1",
        "IRS1",
        "G6PD",
        "EEF1A1",
        "DUSP6",
        "DUSP4",
        "INHBA",
        "ITGA2",
        "ITPKB",
        "SIX1",
        "STK32C",
        "ABCA3",
        "ASAH1",
        "BPGM",
        "CDC25B",
        "COL15A1",
        "DPYSL3",
        "F3",
        "FUCA1",
        "IFI6",
        "GLS",
        "HAGH",
        "HAS2",
        "INSIG1",
        "STMN1",
        "MANBA",
        "MDK",
        "PCOLCE",
        "SERPINF1",
        "SERPINI1",
        "PLAT",
        "PLOD2",
        "PPP1R3C",
        "PTMA",
        "SCD",
        "SFRP1",
        "SLC1A1",
        "TAP1",
        "KHSRP",
        "CREG1",
        "GGH",
        "SYNGR2",
        "SLIT2",
        "AKAP12",
        "ISG15",
        "ATP6AP2",
        "MLLT11",
        "CBX5",
        "ORC6",
        "TUBG2",
        "OSTM1",
        "RAB6B",
        "DHRS7",
        "TM7SF3",
        "GNG2",
        "USP53",
        "HERC6",
        "SULF2",
        "EMC7",
        "OLFML3",
        "NCEH1",
        "NABP1",
        "COLEC12",
        "SLC9A7",
        "LMBRD2",
        "FANK1",
        "ATG4A",
        "WDR63",
        "FAM43A",
        "DOCK11",
        "PRICKLE2",
        "PGM2L1",
        "SLC46A3",
        "IGIP",
    ],
    "SID6": [
        "MYC",
        "AURKB",
        "CCND1",
        "CDKN1A",
        "CDKN2B",
        "ID1",
        "IGFBP3",
        "IGFBP5",
        "LMNA",
        "MKI67",
        "MMP2",
        "STAT1",
        "TWIST1",
        "PCNA",
        "CCNE1",
        "TBX2",
        "PRKCD",
        "ARHGAP18",
        "WNT5A",
        "UBA52",
        "TAGLN",
        "SLC13A3",
        "RRM2",
        "RRM1",
        "RPS27A",
        "RPL5",
        "PSMB5",
        "MAPK10",
        "ERRFI1",
        "PGD",
        "MYBL2",
        "SMAD6",
        "SMAD3",
        "SMAD1",
        "JUN",
        "CDC26",
        "ETS2",
        "ENG",
        "EEF1B2",
        "DUSP4",
        "E2F7",
        "JDP2",
        "CLU",
        "CKB",
        "UBE2C",
        "CENPA",
        "ANAPC10",
        "CD9",
        "CSF1",
        "CXCL12",
        "FAS",
        "IGFBP6",
        "VEGFA",
        "BHLHE40",
        "CDK2AP1",
        "DDB2",
        "DHCR24",
        "LGALS3",
        "NUAK1",
        "PMVK",
        "SIX1",
        "YAP1",
        "ZFP36",
        "ABCA3",
        "ANGPT1",
        "SLC25A5",
        "ANXA4",
        "CDC20",
        "CENPE",
        "CENPF",
        "CKS2",
        "COL6A3",
        "COL8A1",
        "DAB2",
        "DGKA",
        "GADD45A",
        "DNASE1L1",
        "DPYSL3",
        "EMP2",
        "EPAS1",
        "EPHA4",
        "CLN8",
        "ERH",
        "FBL",
        "ETV4",
        "FANCA",
        "FHL1",
        "IFI6",
        "GAS6",
        "HAS2",
        "HLA-B",
        "HMGB3",
        "HMMR",
        "HNRNPH1",
        "SP110",
        "IFIT1",
        "KIF11",
        "LAMC1",
        "STMN1",
        "LIPA",
        "LOXL2",
        "LSAMP",
        "MDK",
        "MEST",
        "ODC1",
        "PAM",
        "ENPP2",
        "PLCB4",
        "PRIM1",
        "PRRG1",
        "HTRA1",
        "RPL4",
        "RPS9",
        "RPS16",
        "SORT1",
        "SHMT2",
        "SLC1A1",
        "SOX4",
        "THBS1",
        "TPM2",
        "TTK",
        "BTG2",
        "SLC7A5",
        "ENC1",
        "PEA15",
        "ADAM19",
        "GGH",
        "SPHK1",
        "P4HA2",
        "PAPSS2",
        "SYNGR2",
        "SYNGR1",
        "TRIP13",
        "AKAP12",
        "ISG15",
        "HDAC9",
        "MELK",
        "GINS1",
        "GFPT2",
        "AASS",
        "TRIM22",
        "NDC80",
        "IFITM3",
        "SEMA3C",
        "RAD51AP1",
        "FILIP1L",
        "OIP5",
        "COBLL1",
        "MAPRE3",
        "KIAA0930",
        "NCAPH",
        "DDAH1",
        "DDX58",
        "APOL2",
        "RAD54B",
        "POC1A",
        "CNRIP1",
        "CLIP3",
        "TMEM251",
        "SLC17A5",
        "CYFIP2",
        "DKK3",
        "TNFRSF21",
        "UBE2T",
        "LRP12",
        "RRM2B",
        "ANGPTL4",
        "NUSAP1",
        "SCARA3",
        "DPH5",
        "COQ3",
        "MIS18A",
        "GNG2",
        "RIN2",
        "SAMD9",
        "HAUS4",
        "HERC6",
        "ZWILCH",
        "CDCA8",
        "CEP55",
        "MNS1",
        "MCM10",
        "CDCA7L",
        "PLXNA3",
        "HHAT",
        "NXT2",
        "SULF2",
        "OLFML3",
        "PRTFDC1",
        "RCN3",
        "C3orf14",
        "NCEH1",
        "LSM2",
        "CLSPN",
        "CENPK",
        "NABP1",
        "NT5DC2",
        "CENPH",
        "NUP85",
        "KIF18A",
        "NUF2",
        "ANTXR1",
        "SLC9A7",
        "NEXN",
        "ZNF300",
        "HTRA3",
        "FBXO32",
        "RMI2",
        "WDR63",
        "LACC1",
        "CKAP2L",
        "ARL6IP6",
        "CDCA2",
        "ITPRIPL2",
        "ZNF610",
        "PRICKLE2",
        "SAMD9L",
        "HYLS1",
        "ASPM",
        "GAS2L3",
        "FAM111B",
        "CENPW",
        "ARHGEF37",
        "SHC4",
    ],
}

In [ ]:
gene_sets = [set(g) for g in sid_genes.values()]
for g in gene_sets:
    print(len(g))
sid_set = set.union(*gene_sets)
print("\n")
print(len(sid_set))

In [13]:
shared_cell_frac = 0.7
seed = 69

rng = np.random.default_rng(seed)


# Гарантируем CSR-формат
if sp.issparse(adata.X):
    if not sp.isspmatrix_csr(adata.X):
        adata.X = adata.X.tocsr()
else:
    raise TypeError("adata.X is not a sparse matrix")

Код ниже выполняет фильтрацию и нормировку (даунсэмплинг) данных

Цель — выровнять глубину секвенирования (число экспрессированных генов) между клетками одного типа. 

Таргетное число генов для кл. типа:
1. 75 квантиль по GSE_id 
2. веса GSE_id: $ \text{Weight}_{GSE} = \sqrt[3]{n\_cells} \cdot n\_unique\_gsm $
3. target_val = взвешенная медиана от 75 квантиля, округляем до целого


shared_genes - экспрессируются в shared_cell_frac всех клеток в кл типе. Они сохраняются при редукции. Остальные гены удаляются случайно до target_val

In [52]:
filtered_adatas = []
downsampling_stats = []

for t in tqdm(sorted(adata.obs.Tissue_global.unique()), desc="Tissues"):
    adata_t = adata[adata.obs.Tissue_global == t].copy()
    for ct in tqdm(sorted(adata_t.obs.cell_type.unique()), desc=f"{t}", leave=False):
        adata_ct = adata_t[adata_t.obs.cell_type == ct].copy()

        n_before = np.asarray(adata_ct.X.getnnz(axis=1)).ravel()

        # делаем DataFrame для удобства
        df = pd.DataFrame(
            {
                "n_expr_genes": adata_ct.X.getnnz(axis=1),
                "GSE_id": adata_ct.obs["GSE_id"].values,
                "GSM_id": adata_ct.obs["GSM_id"].values,
            }
        )

        # группируем по GSE_id и сразу фильтруем
        gse_stats = df.groupby("GSE_id", observed=True).agg(
            n_unique_gsm=("GSM_id", "nunique"),
            n_cells=("n_expr_genes", "size"),
            q75=("n_expr_genes", lambda x: x.quantile(0.75)),
        )
        # фильтр: >= 100 клеток в серии - сделан ранее

        # вычисляем вес и target_val
        gse_stats["weight"] = np.cbrt(gse_stats["n_cells"]) * gse_stats["n_unique_gsm"]
        gse_stats = gse_stats.sort_values(by="weight", ascending=False)

        w = wquantiles.median(gse_stats["q75"].values, gse_stats["weight"].values)
        target_val = int(np.ceil(w))

        # доля клеток, в которых ген экспрессируется, и список генов
        gene_frac = adata_ct.X.getnnz(axis=0) / adata_ct.n_obs
        shared_genes_list = adata_ct.var_names[gene_frac >= shared_cell_frac]
        shared_gene_mask = adata_ct.var_names.isin(shared_genes_list)

        print(
            f"[{t} / {ct}] shared_genes={len(shared_genes_list)}, target = {target_val}",
            file=sys.stderr,
        )

        # ---------------------------------------------------------
        # Downsampling
        # ---------------------------------------------------------
        X = adata_ct.X.tocsr()

        # Работаем непосредственно с CSR internals
        for i in range(X.shape[0]):
            start = X.indptr[i]
            end = X.indptr[i + 1]
            # Индексы и значения генов, выраженных в клетке
            expressed_idx = X.indices[start:end]
            # Сколько shared genes есть в этой клетке
            is_shared = shared_gene_mask[expressed_idx]

            shared_pos = np.flatnonzero(is_shared)
            other_pos = np.flatnonzero(~is_shared)

            n_needed = target_val - len(shared_pos)

            # ---------------------------------------------
            # Выбираем, какие позиции оставить
            # ---------------------------------------------

            if n_needed <= 0:
                # Теоретически shared genes >= target. Shared genes имеют приоритет.
                keep_pos = shared_pos
            elif len(other_pos) > n_needed:
                # Все shared + случайная выборка остальных
                kept_other_pos = rng.choice(other_pos, size=n_needed, replace=False)
                keep_pos = np.concatenate([shared_pos, kept_other_pos])

            else:
                # В клетке и так недостаточно генов для достижения target_val
                keep_pos = np.concatenate([shared_pos, other_pos])

            # ---------------------------------------------
            # Обнуляем ненужные значения напрямую в X.data
            # ---------------------------------------------

            remove_mask = np.ones(end - start, dtype=bool)
            remove_mask[keep_pos] = False
            X.data[start:end][remove_mask] = 0

        # Физически удаляем нулевые элементы из CSR
        X.eliminate_zeros()

        adata_ct.X = X

        n_after = np.asarray(adata_ct.X.getnnz(axis=1)).ravel()

        # ---------------------------------------------------------
        # Статистики
        # ---------------------------------------------------------
        before_stats = (
            pd.Series(n_before).describe(percentiles=[0.25, 0.50, 0.75]).astype("int")
        )
        after_stats = (
            pd.Series(n_after).describe(percentiles=[0.25, 0.50, 0.75]).astype("int")
        )

        changed = n_before != n_after

        downsampling_stats.append(
            {
                "Tissue": t,
                "cell_type": ct,
                "n_cells": len(n_before),
                "target_val": target_val,
                "n_shared_genes": len(shared_genes_list),
                # BEFORE
                "before_min": before_stats["min"],
                "before_q25": before_stats["25%"],
                "before_median": before_stats["50%"],
                "before_mean": before_stats["mean"],
                "before_q75": before_stats["75%"],
                "before_max": before_stats["max"],
                "before_std": before_stats["std"],
                # AFTER
                "after_min": after_stats["min"],
                "after_q25": after_stats["25%"],
                "after_median": after_stats["50%"],
                "after_mean": after_stats["mean"],
                "after_q75": after_stats["75%"],
                "after_max": after_stats["max"],
                "after_std": after_stats["std"],
                # CHANGE
                "n_cells_changed": changed.sum(),
                "frac_cells_changed": changed.mean(),
                "mean_genes_removed": np.mean(n_before - n_after),
                "median_genes_removed": np.median(n_before - n_after),
            }
        )

        filtered_adatas.append(adata_ct)

Tissues:   0%|                                                                                    | 0/7 [00:00<?, ?it/s]

[Colon / B cells] shared_genes=152, target = 1373
[Colon / CD4 T cells] shared_genes=131, target = 1317
[Colon / CD8 T cells] shared_genes=166, target = 1409
[Colon / Crypt stem cells] shared_genes=1084, target = 4923
[Colon / Enterocytes] shared_genes=254, target = 3163
[Colon / Fibroblasts] shared_genes=263, target = 2744
[Colon / Goblet cells] shared_genes=366, target = 3358
[Colon / Mast cells] shared_genes=173, target = 1636
[Colon / Mid-crypt cells] shared_genes=1004, target = 5108
[Colon / Plasma cells] shared_genes=236, target = 1852
[Colon / γδ T cells] shared_genes=264, target = 1952
Tissues:  14%|██████████▊                                                                 | 1/7 [00:08<00:50,  8.45s/it][Heart_Cell / CD4 T cells] shared_genes=165, target = 1090
[Heart_Cell / CD8 T cells] shared_genes=168, target = 1112
[Heart_Cell / Endothelial cells] shared_genes=200, target = 1694
[Heart_Cell / Fibroblasts] shared_genes=241, target = 2073
[Heart_Cell / Monocytes/macrophages] 

In [53]:
final_adata = sc.concat(filtered_adatas, index_unique=None)
downsampling_stats_df = pd.DataFrame(downsampling_stats)

In [54]:
adata

AnnData object with n_obs × n_vars = 1253024 × 43059
    obs: 'GSE_id', 'GSM_id', 'Age', 'Tissue_global', 'cell_type_final', 'Age_bin', 'cell_type', 'n_expr_genes'
    var: 'n_cells'
    uns: 'Tissue_global_colors'
    obsm: 'X_scANVI', 'X_scVI', 'X_umap', '_scvi_extra_categorical_covs'

In [55]:
final_adata

AnnData object with n_obs × n_vars = 1253024 × 43059
    obs: 'GSE_id', 'GSM_id', 'Age', 'Tissue_global', 'cell_type_final', 'Age_bin', 'cell_type', 'n_expr_genes'
    obsm: 'X_scANVI', 'X_scVI', 'X_umap', '_scvi_extra_categorical_covs'

In [56]:
with pd.option_context("display.max_rows", None):
    # Число ненулевых генов в каждой клетке
    n_detected_genes = np.asarray((final_adata.X > 0).sum(axis=1)).ravel()

    # Временный DataFrame
    qc_df = final_adata.obs[["Tissue_global", "cell_type"]].copy()
    qc_df["n_detected_genes"] = n_detected_genes

    # Статистики для каждой пары Tissue × cell_type
    gene_stats = qc_df.groupby(["Tissue_global", "cell_type"], observed=True)[
        "n_detected_genes"
    ].describe(percentiles=[0.25, 0.50, 0.75])

    display(gene_stats.astype("int"))

count  mean   std  min   25%   50%   75%   max
Tissue_global cell_type                                                                      
Colon         B cells                           3948  1068   258  511   863  1058  1373  1373
              CD4 T cells                       6250  1039   272  503   796  1080  1317  1317
              CD8 T cells                       5018  1135   282  508   884  1207  1409  1409
              Crypt stem cells                 15651  3624  1180  525  2885  3756  4911  4923
              Enterocytes                      15321  2084   886  502  1287  2107  3133  3163
              Fibroblasts                       4262  2032   684  507  1467  2134  2744  2744
              Goblet cells                      7801  2352   881  504  1630  2438  3358  3358
              Mast cells                         334  1199   369  555   876  1205  1636  1636
              Mid-crypt cells                   1458  3762  1421  543  2889  4224  5107  5108
              Plasma cells                     20100  1433   390  502  1122  1474  1852  1852
              γδ T cells                        1457  1627   390  530  1393  1792  1952  1952
Heart_Cell    CD4 T cells                       1483   831   207  500   637   826  1089  1090
              CD8 T cells                       1815   861   204  501   680   855  1112  1112
              Endothelial cells                71050  1183   408  500   811  1161  1694  1694
              Fibroblasts                       2778  1484   497  500  1083  1511  2072  2073
              Monocytes/macrophages            10523  1188   388  500   840  1182  1664  1664
              Mural cells                      20566  1126   349  500   819  1104  1562  1562
              NK cells                          4121   881   202  500   707   896  1121  1121
Heart_Nuclei  Atrial cardiomyocytes            38152  1704   674  500  1107  1679  2540  2540
              CD4 T cells                       2790   963   257  500   732   987  1263  1264
              CD8 T cells                       3674   887   230  500   675   888  1167  1167
              Endothelial cells                41333   982   349  500   655   901  1452  1452
              Fibroblasts                     118321  1235   507  500   750  1160  1891  1891
              Lymphatic endothelium              528  1382   626  500   695  1421  2132  2135
              Mast cells                        1240   700   119  500   592   700   848   848
              Monocytes/macrophages            29421  1474   604  500   900  1483  2207  2207
              Mural cells                      61507   845   207  500   655   843  1102  1102
              NK cells                          1621   975   274  501   746   958  1328  1328
              Ventricular cardiomyocytes      174929  1689   591  500  1190  1709  2395  2395
Ileum         B cells                           7527  1481   459  505  1108  1528  1996  2011
              CD4 T cells                       6810  1117   318  502   838  1155  1485  1486
              CD8 T cells                       7302  1283   377  503   966  1324  1715  1716
              Crypt stem cells                  5238  2684  1316  512  1517  2685  4290  4290
              Endothelial cells                  438  2324   861  535  1567  2340  3363  3367
              Enterocytes                      45157  2385   821  505  1766  2436  3311  3351
              Fibroblasts                        588  1817   578  530  1392  1833  2469  2471
              Goblet cells                      9542  1947   771  504  1310  1920  2835  2920
              Monocytes/macrophages              350  2184   859  538  1450  2385  3082  3084
              Paneth cells                      2407  2862  1213  531  1889  3030  4125  4267
              Plasma cells                      2838  1358   431  512   996  1422  1846  1846
              γδ T cells                         555  1464   341  516  1233  1489  1844  1847
Kidne

In [58]:
print(gene_stats["25%"].mean())
print(gene_stats["25%"].median())

1092.0069444444443
968.75


In [59]:
clean_adata = ad.AnnData(obs=final_adata.obs, X=final_adata.X, var=final_adata.var)
# тут прямые ссылки, с новым объектом не работать!!!
# либо сделать .copy()

In [ ]:
clean_adata.write(".././0.1_adata_downsampled.h5ad")

In [ ]:
downsampling_stats_df.head()

,Tissue,cell_type,n_cells,target_val,n_shared_genes,before_min,before_q25,before_median,before_mean,before_q75,...,after_q25,after_median,after_mean,after_q75,after_max,after_std,n_cells_changed,frac_cells_changed,mean_genes_removed,median_genes_removed
0,Colon,B cells,3948,1373,152,511,863,1058,1216,1457,...,863,1058,1068,1373,1373,258,1125,0.284954,147.815603,0.0
1,Colon,CD4 T cells,6250,1317,131,503,796,1080,1225,1563,...,796,1080,1039,1317,1317,272,2304,0.368640,186.247200,0.0
2,Colon,CD8 T cells,5018,1409,166,508,884,1207,1333,1681,...,884,1207,1135,1409,1409,282,1934,0.385413,198.017935,0.0
3,Colon,Crypt stem cells,15651,4923,1084,525,2885,3756,3937,4911,...,2885,3756,3624,4911,4923,1180,3887,0.248355,312.435563,0.0
4,Colon,Enterocytes,15321,3163,254,502,1287,2107,2376,3133,...,1287,2107,2084,3133,3163,886,3717,0.242608,292.433262,0.0


In [ ]:
downsampling_stats_df.to_csv(".././0.1_adata_dw_describe.csv")